# Agents & Tools — The ReAct Agent

**Phase 05 · Flagship Notebook**

---

## ⚠️ Prerequisites

| Requirement | How to satisfy |
|---|---|
| Phase 03 complete | Run `rag_pipeline.ipynb` to generate `support_tickets.csv` and build `chroma_db/` |
| Phase 07 complete | Run `dbt seed && dbt run` inside `dbt_pipeline/ai_learning/` to create `ai_learning.duckdb` |
| **Ollama running** | `ollama serve` in a terminal, with `llama3` pulled (`ollama pull llama3`) |
| **Dependencies** | `pip install langgraph langchain-core langchain-ollama duckdb` |

```
pip install langgraph langchain-core langchain-ollama duckdb
```


---

## Section 1 — What is an Agent?

### The difference between a chain and an agent

Everything you have built up to Phase 04 is a **chain**: a fixed sequence of steps
executed in a predetermined order.

```
Chain:  User Query → Retriever → Prompt → LLM → Answer
        (same path every time, no decisions)
```

An **agent** is different. It is a loop that lets the LLM *decide* what to do next:
which tool to call, with what arguments, and whether to stop.

```
Agent:  User Query → LLM reasons about what to do next
                        ↓
                   Call a tool? → Execute tool → Observe result
                        ↓                              ↓
                   Final answer ←────────────────── Enough info?
```

### The ReAct loop

**ReAct** stands for **Re**ason + **Act**. Published by Yao et al. (2022), it is the
algorithm that most production agents use today.

```
┌─────────────────────────────────────────────────────────────────────┐
│                         ReAct Loop                                  │
│                                                                     │
│   ┌──────────┐      ┌──────────┐      ┌──────────┐                 │
│   │  REASON  │─────▶│   ACT    │─────▶│ OBSERVE  │                 │
│   │          │      │          │      │          │                 │
│   │ LLM looks│      │ Execute  │      │ Tool     │                 │
│   │ at state │      │ the tool │      │ result   │                 │
│   │ and picks│      │ call     │      │ added to │                 │
│   │ next tool│      │ (or stop)│      │ context  │                 │
│   └──────────┘      └──────────┘      └────┬─────┘                 │
│        ▲                                    │                       │
│        └────────────────────────────────────┘  (repeat until done) │
│                                                                     │
│   When the LLM decides it has enough information it emits a         │
│   final answer instead of another tool call — the loop ends.        │
└─────────────────────────────────────────────────────────────────────┘
```

Each iteration adds new information to the **context window**. The LLM sees the
entire history — question, tool calls made, results observed — and uses all of it
when deciding what to do next.

### Why agents matter for the SPK stack

In an AI-augmented data platform:

- The **S** (Spark / structured data) is accessed via tools like `query_dbt_gold`
- The **P** (Prefect / pipelines) can be triggered by agents as orchestration actions
- The **K** (Knowledge / vector stores) is accessed via `search_support_tickets`

An agent that can query all three layers can answer questions a static RAG pipeline
cannot — it routes the question to the right data source automatically.


---

## Section 2 — Tools

### What is a tool?

A tool is a **Python function that the agent is allowed to call**. The agent
(the LLM) does not execute tools itself — it *requests* a tool call by emitting
structured JSON. The framework executes the real Python function and feeds the
result back into the context.

```
Agent (LLM) says:  { "tool": "lookup_product", "args": {"product_id": "P001"} }
Framework runs:    lookup_product(product_id="P001")
Framework feeds:   "Product: Wireless Headphones, price: $79.99 ..." back to LLM
```

**The docstring is the tool description** — LangGraph passes it to the LLM so
the model knows when and how to use each tool. Write docstrings as if you are
explaining the tool to a capable but uninformed colleague.

We define **3 tools** in this section:

| Tool | Data source | What it answers |
|---|---|---|
| `search_support_tickets` | Chroma vector store (Phase 03) | Semantic search over ticket text |
| `calculate` | Python `ast` module | Safe arithmetic expressions |
| `lookup_product` | `products.csv` seed file | Product details by ID |


In [ ]:
import os
import ast
import math
import operator
import warnings
import pandas as pd
warnings.filterwarnings('ignore')

NOTEBOOK_DIR  = os.path.abspath('')
DATA_DIR      = os.path.join(NOTEBOOK_DIR, '..', '..', 'data', 'raw')
CHROMA_DIR    = os.path.join(NOTEBOOK_DIR, '..', '03_rag_pipeline', 'chroma_db')
PRODUCTS_PATH = os.path.join(
    NOTEBOOK_DIR, '..', '..', 'dbt_pipeline', 'ai_learning', 'seeds', 'products.csv'
)
DBT_DUCKDB    = os.path.join(
    NOTEBOOK_DIR, '..', '..', 'dbt_pipeline', 'data', 'ai_learning.duckdb'
)

print('Paths configured:')
print(f'  CHROMA_DIR    : {os.path.abspath(CHROMA_DIR)}')
print(f'  PRODUCTS_PATH : {os.path.abspath(PRODUCTS_PATH)}')
print(f'  DBT_DUCKDB    : {os.path.abspath(DBT_DUCKDB)}')


In [ ]:
# ── Install dependencies if not present ─────────────────────────────────
import subprocess, sys

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        print(f'Installing {package} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

install_if_missing('langgraph')
install_if_missing('langchain-core', 'langchain_core')
install_if_missing('duckdb')
install_if_missing('langchain-ollama', 'langchain_ollama')
print('All dependencies ready.')


In [ ]:
# ── Tool 1: search_support_tickets ───────────────────────────────────────
# Lazily initialises the Chroma vector store on first call.
# If the Phase 03 chroma_db/ directory does not exist, auto-generates
# synthetic tickets so the agent works in isolation.

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

_vectorstore = None   # module-level cache

def _get_vectorstore():
    global _vectorstore
    if _vectorstore is not None:
        return _vectorstore

    embed = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

    if os.path.isdir(CHROMA_DIR):
        _vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embed)
        print(f'Loaded Chroma from {CHROMA_DIR}')
    else:
        # Auto-generate 30 synthetic tickets so the notebook works standalone
        from langchain.text_splitter import RecursiveCharacterTextSplitter
        synthetic = [
            'User cannot log in after password reset. Error: invalid credentials. Priority: high.',
            'Login page returns 500 error for all users on Chrome. Investigated — session cookie bug.',
            'Multiple users report login failures after the 2.3.1 deploy. Rolling back.',
            'Billing page shows incorrect charge amount. Customer charged twice. Refund initiated.',
            'Invoice download fails with 404. Backend S3 bucket permissions misconfigured.',
            'Customer billed for cancelled subscription. Accounting notified.',
            'Payment gateway timeout causing duplicate charges. Stripe webhook issue.',
            'Product search returns no results for electronics category after index rebuild.',
            'API rate limit errors appearing in production logs. Spike in /search endpoint.',
            'Database connection pool exhausted during peak hours. Scaling event triggered.',
            'Export CSV feature broken — file contains only headers, no rows.',
            'Email notifications not sent after order confirmation. SMTP relay down.',
            'Mobile app crashes on Android 14 when opening account settings.',
            'Two-factor authentication SMS not delivered to international numbers.',
            'Admin dashboard charts render blank after timezone update.',
            'Password reset email not arriving. SPF record misconfigured.',
            'Login with Google OAuth broken after client ID rotation.',
            'Session timeout too aggressive — users logged out after 5 minutes.',
            'Billing statement PDF shows wrong currency symbol for EU customers.',
            'Subscription upgrade not reflected in account portal for 24 hours.',
            'User reports billing discrepancy: charged $99 instead of $9. Pricing bug.',
            'Search autocomplete suggestions show deleted products.',
            'High memory usage on app servers correlates with /export endpoint usage.',
            'Users cannot upload profile images larger than 1MB. Limit undocumented.',
            'Webhook delivery failing for Slack integration after cert expiry.',
            'Login works but dashboard shows empty — permissions caching bug.',
            'Automated billing email subject line has wrong month name.',
            'Forgot password flow redirects to 404 page after link click.',
            'Product rating not updating after new review submitted.',
            'Support ticket creation fails when message body exceeds 5000 chars.',
        ]
        splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
        docs = splitter.create_documents(synthetic)
        _vectorstore = Chroma.from_documents(docs, embed)
        print('Built in-memory Chroma from synthetic tickets (Phase 03 chroma_db not found).')

    return _vectorstore


def search_support_tickets(query: str) -> str:
    """
    Search the support ticket knowledge base for tickets semantically similar
    to the given query. Returns the top 5 most relevant ticket excerpts as
    a numbered list. Use this tool whenever the user asks about support issues,
    customer complaints, bugs, or incidents.

    Args:
        query: A natural language search query describing what to find.

    Returns:
        A string containing the top 5 matching ticket excerpts.
    """
    vs = _get_vectorstore()
    results = vs.similarity_search(query, k=5)
    if not results:
        return 'No relevant tickets found.'
    lines = [f'{i+1}. {doc.page_content.strip()}' for i, doc in enumerate(results)]
    return '\n'.join(lines)


# Quick smoke-test
print(search_support_tickets('login problems'))


In [ ]:
# ── Tool 2: calculate ────────────────────────────────────────────────────
# Evaluates a safe arithmetic expression using AST parsing.
# We do NOT use eval() — it is a security risk.

_SAFE_OPS = {
    ast.Add:  operator.add,
    ast.Sub:  operator.sub,
    ast.Mult: operator.mul,
    ast.Div:  operator.truediv,
    ast.Pow:  operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def _safe_eval(node):
    """Recursively evaluate an AST node using only safe numeric operations."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp):
        op_fn = _SAFE_OPS.get(type(node.op))
        if op_fn is None:
            raise ValueError(f'Unsupported operator: {type(node.op).__name__}')
        return op_fn(_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):
        op_fn = _SAFE_OPS.get(type(node.op))
        if op_fn is None:
            raise ValueError(f'Unsupported operator: {type(node.op).__name__}')
        return op_fn(_safe_eval(node.operand))
    raise ValueError(f'Unsupported expression node: {type(node).__name__}')


def calculate(expression: str) -> str:
    """
    Safely evaluate a mathematical expression and return the numeric result.
    Supports +, -, *, /, ** (exponentiation). Do NOT pass Python code —
    only arithmetic expressions like '15 / 100 * 67.49' or '(12 + 8) / 4'.

    Args:
        expression: A string containing a numeric arithmetic expression.

    Returns:
        The result as a string, or an error message if the expression is invalid.
    """
    try:
        tree = ast.parse(expression, mode='eval')
        result = _safe_eval(tree.body)
        return str(round(result, 6))
    except Exception as e:
        return f'Error evaluating expression: {e}'


# Quick smoke-tests
print(calculate('15 / 100 * 67.49'))    # 15% of $67.49
print(calculate('(12 + 8) / 4'))        # 5.0
print(calculate('2 ** 10'))             # 1024
print(calculate('__import__("os")'))   # should error safely


In [ ]:
# ── Tool 3: lookup_product ───────────────────────────────────────────────
# Reads products.csv (the dbt seed file) and returns a single product row.
# Accepts integer IDs (1, 2, ...) or string IDs with a P-prefix (P001, ...).

_products_df = None

def _get_products():
    global _products_df
    if _products_df is None:
        _products_df = pd.read_csv(PRODUCTS_PATH)
    return _products_df


def lookup_product(product_id: str) -> str:
    """
    Look up a product by its ID from the product catalogue. Accepts an integer
    ID (e.g. '1') or a zero-padded string ID (e.g. 'P001' or '001').
    Returns the product name, category, price, rating, review count, and status.
    Use this tool when the user asks about a specific product or its attributes.

    Args:
        product_id: The product identifier as a string (e.g. '1', 'P001').

    Returns:
        A formatted string describing the product, or a not-found message.
    """
    df = _get_products()

    # Normalise: strip leading 'P' or 'p' and leading zeros
    clean = product_id.strip().lstrip('Pp').lstrip('0') or '0'
    try:
        pid = int(clean)
    except ValueError:
        return f'Invalid product ID: {product_id!r}'

    row = df[df['product_id'] == pid]
    if row.empty:
        return f'No product found with ID {product_id}.'

    r = row.iloc[0]
    return (
        f'Product ID   : {r.product_id}\n'
        f'Name         : {r["name"]}\n'
        f'Category     : {r.category}\n'
        f'Price        : ${r.price:.2f}\n'
        f'Rating       : {r.rating} / 5.0\n'
        f'Review count : {int(r.review_count)}\n'
        f'Status       : {r.status}'
    )


# Smoke-tests
print(lookup_product('1'))
print()
print(lookup_product('P011'))
print()
print(lookup_product('P099'))   # not found


---

## Section 3 — Build a ReAct Agent with LangGraph

### LangGraph: agents as state machines

**LangGraph** models an agent as a directed graph where:

- **Nodes** are actions (call the LLM, execute a tool)
- **Edges** are decisions (should we call a tool or return a final answer?)
- **State** is a typed dictionary passed between nodes on every step

The ReAct loop maps directly onto this graph:

```
START
  │
  ▼
[agent] ──── LLM decides to call a tool ────▶ [tools] ──┐
  ▲                                                       │
  └───────────────────────────────────────────────────────┘
  │
  └── LLM decides it has enough info ──▶ END
```

### Key components

| Component | Role |
|---|---|
| `ChatOllama` | The LLM — reasoning happens here |
| `bind_tools()` | Tells the LLM which tools exist and what they do |
| `ToolNode` | Executes whichever tool the LLM requested |
| `MessagesState` | The state — a list of messages (HumanMessage, AIMessage, ToolMessage) |
| `should_continue` | Conditional edge: look at the last message to decide next step |


In [ ]:
from langchain_core.tools import tool as lc_tool
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from typing import Annotated
from typing_extensions import TypedDict

# ── Wrap plain Python functions as LangChain tools ────────────────────────
# The @lc_tool decorator reads the docstring and function signature to build
# the JSON schema the LLM uses to decide when/how to call each tool.

@lc_tool
def tool_search_support_tickets(query: str) -> str:
    """Search the support ticket knowledge base for tickets semantically
    similar to the given query. Returns the top 5 most relevant ticket
    excerpts as a numbered list. Use this when the user asks about support
    issues, customer complaints, bugs, or incidents.

    Args:
        query: A natural language search query.
    """
    return search_support_tickets(query)


@lc_tool
def tool_calculate(expression: str) -> str:
    """Safely evaluate a mathematical expression. Supports +, -, *, /, **.
    Example: '15 / 100 * 67.49' returns '10.1235'. Do NOT pass Python code.

    Args:
        expression: A string arithmetic expression.
    """
    return calculate(expression)


@lc_tool
def tool_lookup_product(product_id: str) -> str:
    """Look up a product by its integer ID from the product catalogue.
    Returns name, category, price, rating, review count, and status.
    Use this when the user asks about a specific product.

    Args:
        product_id: The product ID as a string (e.g. '1', 'P001').
    """
    return lookup_product(product_id)


TOOLS = [tool_search_support_tickets, tool_calculate, tool_lookup_product]
print('Tools registered:')
for t in TOOLS:
    print(f'  {t.name}: {t.description[:80]}')


In [ ]:
# ── Define agent state ────────────────────────────────────────────────────
# AgentState holds the conversation as a list of messages.
# The `add_messages` reducer appends new messages rather than replacing them.

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# ── LLM with tools bound ──────────────────────────────────────────────────
llm = ChatOllama(model='llama3', temperature=0)
llm_with_tools = llm.bind_tools(TOOLS)


# ── Node: agent ───────────────────────────────────────────────────────────
def agent_node(state: AgentState) -> AgentState:
    """Call the LLM with the current message history. The LLM may either
    request a tool call (AIMessage with tool_calls) or produce a final
    answer (AIMessage without tool_calls).
    """
    response = llm_with_tools.invoke(state['messages'])
    return {'messages': [response]}


# ── Conditional edge: tools or END? ──────────────────────────────────────
def should_continue(state: AgentState) -> str:
    """Return "tools" if the last LLM message has tool calls, else "end"."""
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return 'tools'
    return 'end'


# ── Build and compile the graph ───────────────────────────────────────────
def build_agent_graph(interrupt_tools: bool = False):
    """Build the ReAct agent graph. interrupt_tools=True pauses before tools."""
    tool_node = ToolNode(TOOLS)

    graph = StateGraph(AgentState)
    graph.add_node('agent', agent_node)
    graph.add_node('tools', tool_node)
    graph.set_entry_point('agent')
    graph.add_conditional_edges(
        'agent',
        should_continue,
        {'tools': 'tools', 'end': END},
    )
    graph.add_edge('tools', 'agent')

    interrupt_before = ['tools'] if interrupt_tools else []
    return graph.compile(interrupt_before=interrupt_before)


agent = build_agent_graph()
print('Agent graph compiled successfully.')


In [ ]:
# ── Visualise the state machine ───────────────────────────────────────────
# LangGraph exports a Mermaid diagram of any compiled graph.
# In Jupyter this renders as an SVG; otherwise prints the Mermaid syntax.

try:
    from IPython.display import Image, display
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print('Mermaid diagram (paste into https://mermaid.live/ to visualise):')
    print(agent.get_graph().draw_mermaid())


---

## Section 4 — Run the Agent

### Watching the ReAct loop in action

Each call to `agent.stream()` yields **one event per step**. We print every
step so you can see the full Reason → Act → Observe → Repeat cycle.

Different questions trigger different tool paths:

| Question | Expected tool(s) | Why |
|---|---|---|
| Login problems search | `search_support_tickets` | Semantic search over tickets |
| 15% of avg Electronics price | `lookup_product` × 2 + `calculate` | Multi-step: look up prices, compute |
| Billing ticket summary | `search_support_tickets` | Aggregation over retrieved text |
| P001 product lookup | `lookup_product` | Single tool call |
| Product 6 price + tickets | `lookup_product` + `search_support_tickets` + `calculate` | Three tools |


In [ ]:
def run_agent(question: str, graph=None) -> str:
    """Run the agent on a single question and print every reasoning step."""
    if graph is None:
        graph = agent

    print('=' * 70)
    print(f'QUESTION: {question}')
    print('=' * 70)

    inputs = {'messages': [HumanMessage(content=question)]}
    final_answer = ''

    for step, event in enumerate(graph.stream(inputs, stream_mode='values')):
        last_msg = event['messages'][-1]

        if isinstance(last_msg, HumanMessage):
            print(f'\n[Step {step}] USER: {last_msg.content[:200]}')

        elif isinstance(last_msg, AIMessage):
            if last_msg.tool_calls:
                for tc in last_msg.tool_calls:
                    print(f'\n[Step {step}] AGENT → TOOL CALL: {tc["name"]}')
                    print(f'            args: {str(tc["args"])[:200]}')
            else:
                final_answer = last_msg.content
                print(f'\n[Step {step}] AGENT → FINAL ANSWER:')
                print(final_answer)

        elif isinstance(last_msg, ToolMessage):
            result_preview = last_msg.content[:300].replace('\n', ' | ')
            print(f'\n[Step {step}] TOOL RESULT ({last_msg.name}):')
            print(f'  {result_preview}')

    print('\n' + '-' * 70)
    return final_answer


In [ ]:
# ── Question 1: Semantic search over support tickets ─────────────────────
run_agent('What support tickets mention login problems?')


In [ ]:
# ── Question 2: Multi-step — lookup prices then calculate ────────────────
# The agent looks up two Electronics products, finds their prices,
# then calculates 15% of their average. Requires: lookup x2 + calculate.
run_agent(
    'Look up products 1 and 11 (both Electronics). '
    'What is 15% of their average price?'
)


In [ ]:
# ── Question 3: Summarise billing tickets ────────────────────────────────
run_agent(
    'Find support tickets about billing issues and summarise '
    'the most common type of complaint.'
)


In [ ]:
# ── Question 4: Single product lookup ────────────────────────────────────
run_agent('What is product P001 and how is it rated by customers?')


In [ ]:
# ── Question 5: Multi-tool — lookup + calculate + search ─────────────────
# Requires: lookup_product to get price, calculate for discount,
# search_support_tickets for related complaints. Three tools in one run.
run_agent(
    'What is product 6? Calculate what 20% off its current price would be. '
    'Also, are there any support tickets about this type of product?'
)


---

## Section 5 — Human in the Loop

### Why approve tool calls before execution?

In production systems — especially in the SPK stack — agents may call tools
that have **real-world side effects**:

- Writing to a database
- Sending an email or notification
- Triggering a Prefect pipeline run
- Posting to Slack or an external API

Even read-only tools can be sensitive. A tool that queries a customer's personal
data should require human approval in a GDPR-regulated context.

LangGraph supports this with `interrupt_before=['tools']`: the graph **pauses**
after the LLM decides to call a tool but **before** the tool executes.

```
[agent] decides to call tool_lookup_product(product_id='1')
         │
         ▼
    ⏸  PAUSE — human reviews the proposed tool call
         │
    ✅ approve → [tools] executes → [agent] continues
    ❌ reject  → inject refusal message → [agent] responds gracefully
```

LangGraph uses a **checkpointer** to persist graph state across the pause.
In production you would use a PostgreSQL or Redis checkpointer.
Here we use the built-in `MemorySaver` for simplicity.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

def build_hitl_graph():
    """Build a ReAct agent that pauses before every tool execution."""
    tool_node = ToolNode(TOOLS)
    memory    = MemorySaver()

    graph = StateGraph(AgentState)
    graph.add_node('agent', agent_node)
    graph.add_node('tools', tool_node)
    graph.set_entry_point('agent')
    graph.add_conditional_edges(
        'agent',
        should_continue,
        {'tools': 'tools', 'end': END},
    )
    graph.add_edge('tools', 'agent')

    # interrupt_before=['tools'] pauses BEFORE the tools node runs
    return graph.compile(checkpointer=memory, interrupt_before=['tools'])


hitl_agent = build_hitl_graph()
print('Human-in-the-loop agent compiled.')


In [ ]:
# ── Run with human approval prompt ───────────────────────────────────────

QUESTION = 'What is product 3 and what is 10% of its price?'
CONFIG   = {'configurable': {'thread_id': 'hitl-demo-1'}}

print('=' * 70)
print(f'QUESTION: {QUESTION}')
print('=' * 70)

inputs = {'messages': [HumanMessage(content=QUESTION)]}

# Phase 1: run until the first interrupt
for event in hitl_agent.stream(inputs, config=CONFIG, stream_mode='values'):
    last = event['messages'][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        tc = last.tool_calls[0]
        print(f'\n⏸  AGENT WANTS TO CALL: {tc["name"]}')
        print(f'   Arguments: {tc["args"]}')
        break

# ── Human approval step ───────────────────────────────────────────────────
# In a real system this would be a UI button or Slack approval message.
# Set APPROVED = False to test the rejection path.
current_state = hitl_agent.get_state(CONFIG)
pending_tools = [
    tc['name']
    for msg in current_state.values['messages']
    if isinstance(msg, AIMessage)
    for tc in (msg.tool_calls or [])
]
print(f'\nPending tool calls: {pending_tools}')

APPROVED = True   # change to False to test rejection

if APPROVED:
    print('✅ Human approved — resuming agent.')
    for event in hitl_agent.stream(None, config=CONFIG, stream_mode='values'):
        last = event['messages'][-1]
        if isinstance(last, ToolMessage):
            print(f'\n[TOOL RESULT] {last.name}: {last.content[:200]}')
        elif isinstance(last, AIMessage) and not last.tool_calls:
            print(f'\n[FINAL ANSWER]: {last.content}')
else:
    print('❌ Human rejected — injecting refusal message.')
    pending_msg = current_state.values['messages'][-1]
    refusal_messages = [
        ToolMessage(
            content='Tool call rejected by human operator.',
            tool_call_id=tc['id'],
            name=tc['name'],
        )
        for tc in pending_msg.tool_calls
    ]
    hitl_agent.update_state(CONFIG, {'messages': refusal_messages})
    for event in hitl_agent.stream(None, config=CONFIG, stream_mode='values'):
        last = event['messages'][-1]
        if isinstance(last, AIMessage) and not last.tool_calls:
            print(f'\n[AGENT RESPONSE AFTER REJECTION]: {last.content}')
            break


---

## Section 6 — Connect the dbt Gold Layer

### Agents + structured data = superpowers

The RAG tools in Sections 2–5 give the agent access to **unstructured** knowledge
(support tickets). The `query_dbt_gold` tool opens a second data layer:
the **aggregated, business-ready tables** built by the dbt pipeline in Phase 07.

```
Unstructured knowledge  →  search_support_tickets  (Chroma / Phase 03)
Structured aggregates   →  query_dbt_gold          (DuckDB / Phase 07)
Raw product data        →  lookup_product          (CSV seed)
```

### Why read-only?

We restrict the tool to `SELECT` statements only. An agent with `INSERT` or `UPDATE`
access to a production database is a significant security risk — it could overwrite
or delete data if the LLM misinterprets a question. The pattern of **read-only
query tools** with **human-in-the-loop for writes** is best practice.

### Gold layer tables available

| Table | Key columns | What it answers |
|---|---|---|
| `main_gold.dept_summary` | department, headcount, avg_salary, avg_performance, avg_tenure_years | HR analytics |
| `main_gold.product_category_summary` | category, product_count, avg_price, avg_rating, total_reviews | Product analytics |

> **Setup:** run `cd dbt_pipeline/ai_learning && dbt seed && dbt run` first.


In [ ]:
# ── Tool 4: query_dbt_gold ────────────────────────────────────────────────
# Opens a read-only DuckDB connection to the gold layer and executes a
# SELECT query. Non-SELECT statements are rejected.

def query_dbt_gold(sql: str) -> str:
    """
    Execute a read-only SQL SELECT query against the dbt gold layer in DuckDB.
    Available tables:
      main_gold.dept_summary (department, headcount, avg_salary, avg_performance, avg_tenure_years)
      main_gold.product_category_summary (category, product_count, avg_price, avg_rating, total_reviews)
    Only SELECT statements are permitted. Do not include semicolons.

    Args:
        sql: A SQL SELECT statement (no semicolons, no DDL/DML).

    Returns:
        Query results as a formatted string table, or an error message.
    """
    stripped = sql.strip().rstrip(';')
    first_word = stripped.split()[0].upper() if stripped.split() else ''
    if first_word != 'SELECT':
        return f'Error: only SELECT queries are permitted. Got: {first_word!r}'

    if not os.path.exists(DBT_DUCKDB):
        return (
            f'DuckDB file not found at {DBT_DUCKDB}. '
            'Please run: cd dbt_pipeline/ai_learning && dbt seed && dbt run'
        )

    try:
        import duckdb
        conn = duckdb.connect(DBT_DUCKDB, read_only=True)
        df = conn.execute(stripped).df()
        conn.close()
        if df.empty:
            return 'Query returned no rows.'
        return df.to_string(index=False)
    except Exception as e:
        return f'Query error: {e}'


# Smoke-test (only works if dbt pipeline has been run)
result = query_dbt_gold('SELECT * FROM main_gold.dept_summary ORDER BY avg_salary DESC')
print(result)


In [ ]:
# ── Register as a LangChain tool and rebuild the agent ───────────────────

@lc_tool
def tool_query_dbt_gold(sql: str) -> str:
    """Execute a read-only SQL SELECT query against the dbt gold layer in DuckDB.
    Available tables:
      main_gold.dept_summary (department, headcount, avg_salary, avg_performance, avg_tenure_years)
      main_gold.product_category_summary (category, product_count, avg_price, avg_rating, total_reviews)
    Only SELECT statements are allowed. Do not include semicolons.

    Args:
        sql: A SQL SELECT statement.
    """
    return query_dbt_gold(sql)


TOOLS_V2 = [
    tool_search_support_tickets,
    tool_calculate,
    tool_lookup_product,
    tool_query_dbt_gold,
]

# Rebuild the agent with all four tools
llm_with_tools_v2 = llm.bind_tools(TOOLS_V2)

def agent_node_v2(state: AgentState) -> AgentState:
    response = llm_with_tools_v2.invoke(state['messages'])
    return {'messages': [response]}

_tool_node_v2 = ToolNode(TOOLS_V2)
_graph_v2     = StateGraph(AgentState)
_graph_v2.add_node('agent', agent_node_v2)
_graph_v2.add_node('tools', _tool_node_v2)
_graph_v2.set_entry_point('agent')
_graph_v2.add_conditional_edges('agent', should_continue, {'tools': 'tools', 'end': END})
_graph_v2.add_edge('tools', 'agent')
agent_v2 = _graph_v2.compile()

print('Agent v2 (with dbt gold layer) compiled.')


In [ ]:
# ── Gold-layer query: highest average salary ─────────────────────────────
run_agent(
    'What department has the highest average salary? '
    'Also tell me how many employees are in that department.',
    graph=agent_v2,
)


In [ ]:
# ── Cross-layer question: product ratings + support tickets ──────────────
run_agent(
    'Which product category has the best average rating? '
    'Are there any support tickets mentioning problems with that category?',
    graph=agent_v2,
)


---

## What You Have Built Across All 6 Phases

| Phase | Topic | Key artifact |
|---|---|---|
| 01 | Embeddings & Similarity | `embeddings_and_similarity.ipynb` — sentence vectors, cosine similarity |
| 02 | LLMs & Prompting | `llms_and_prompting.ipynb` — Ollama, prompt engineering, chat history |
| 03 | RAG Pipeline | `rag_pipeline.ipynb` — Chroma vector store, retrieval-augmented generation |
| 04 | Advanced RAG | `advanced_rag.ipynb` — parent-child chunking, reranking, hybrid BM25+vector, RAGAS |
| 05 | Agents & Tools | `agents_and_tools.ipynb` — ReAct loop, LangGraph, tool use, HITL |
| 07 | dbt Pipeline | `dbt_pipeline/ai_learning/` — Bronze/Silver/Gold medallion in DuckDB |

---

### How this maps to the SPK stack

```
S — Structured data layer
    dbt + DuckDB (Phase 07)
    Bronze → Silver → Gold medallion architecture
    Gold tables are the agent's structured data interface

P — Pipeline orchestration
    Prefect (Phase 06, coming next)
    Agents can trigger pipeline runs as tool calls
    Human-in-the-loop approvals before destructive operations

K — Knowledge layer
    Chroma vector store (Phase 03-04)
    Unstructured knowledge: support tickets, docs, emails
    Semantic search surfaces relevant context for agent reasoning
```

The agent from this notebook sits at the **intersection of all three**:
it routes each question to the right layer and synthesises results into a
single coherent answer.

---

### What to learn next

**MCP Servers (Model Context Protocol)**
- MCP is an open protocol (Anthropic, 2024) that standardises how agents
  discover and call tools across different providers
- Instead of decorating Python functions with `@tool`, you register them
  as MCP tool endpoints that any MCP-compatible agent can discover
- Production dbt, Slack, GitHub, and database connectors already publish
  MCP servers you can drop into your agent without writing glue code

**Knowledge Graphs (Phase 06 preview)**
- A vector store answers: what text is similar to this query?
- A knowledge graph answers: what is the *relationship* between these entities?
- Neo4j + LangChain provides a `GraphCypherQAChain` that lets agents query
  entity relationships the same way they query SQL
- Most production AI platforms combine both: vector for fuzzy semantic search,
  graph for precise relationship traversal

**CI/CD for AI pipelines**
- dbt test runs should be part of every CI pipeline — they are your data quality gate
- RAGAS scores from Phase 04 can be tracked as regression tests:
  if faithfulness drops below 0.7, the pipeline fails
- GitHub Actions + Prefect can run the full test suite on every commit
- Model version pinning (`ollama pull llama3:8b-instruct-q4_K_M`) ensures
  reproducible inference across environments

**Production hardening**
- Replace `MemorySaver` with `PostgresSaver` or `RedisSaver` for persistent
  agent state across restarts
- Add structured output parsing so tool results are validated before the LLM sees them
- Instrument with LangSmith or Langfuse to trace every agent run and debug
  multi-step reasoning failures
- Add rate limiting and cost tracking if swapping Ollama for an API-based LLM
